[Reference](https://suparnachowdhury.medium.com/inside-the-transformer-part-1-embeddings-with-python-f4c2148d1445$0)

The Full Picture at a Glance
- One-hot encoding — Gives each word a unique ID — a single 1 among thousands of zeros. No relationships, every word is an island.
- Word2Vec (2013) — Maps words to geometric coordinates. One vector per word, ignores context.
- Transformer embeddings (2017) — Updates vectors based on surrounding context. Computationally expensive but context-aware.
- Sentence embeddings — Compresses whole sentences into one point in space. Powers semantic search, clustering, and retrieval.


In [2]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 44.9 MB/s eta 0:00:00


In [4]:
import pandas as pd
from gensim.models import Word2Vec
from nltk.tokenize import word_tokenize

df = pd.read_csv('IMDB Dataset.csv')
df.head()

sentences = [word_tokenize(review.lower())
             for review in df['review'][:10000]]

model = Word2Vec(
    sentences,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4
)

model.wv.most_similar('cat')

In [5]:
import gensim.downloader as api
model = api.load("glove-wiki-gigaword-100")

result = model.most_similar(
    positive=["king", "woman"],
    negative=["man"],
    topn=5
)
print(result)

[==================================================] 100.0% 128.1/128.1MB downloaded
[('queen', 0.7698540687561035), ('monarch', 0.6843381524085999), ('throne', 0.6755736470222473), ('daughter', 0.6594556570053101), ('princess', 0.6520534157752991)]


In [6]:
from transformers import AutoTokenizer

model_name = "microsoft/Phi-3-mini-4k-instruct"

prompt = "Winter is cold and snowy."
tokenizer = AutoTokenizer.from_pretrained(model_name)
inputs = tokenizer(prompt)
input_ids = inputs["input_ids"]

print("Token IDs:", input_ids,"\n")

tokens = tokenizer.convert_ids_to_tokens(input_ids)

for token, token_id in zip(tokens, input_ids):
    print(f"{token:15} -> {token_id}")

config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.44k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.94M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

Token IDs: [12267, 338, 11220, 322, 15007, 29891, 29889] 

▁Winter         -> 12267
▁is             -> 338
▁cold           -> 11220
▁and            -> 322
▁snow           -> 15007
y               -> 29891
.               -> 29889


In [8]:
for idx, token in enumerate(tokens):
    word_vector = outputs.last_hidden_state[0, idx, :]
    print(f"{token} embedding (first 4 dims): "
          f"{word_vector[:4].detach().numpy()}")

In [9]:
from transformers import BertTokenizer, BertModel
import torch
import torch.nn.functional as F

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
model = BertModel.from_pretrained("bert-base-uncased")

sentences = [
    "She sat on the river bank.",
    "He went to the bank to deposit his check.",
    "We had a picnic on the bank near the stream."
]

bank_vectors = []

for sentence in sentences:
    inputs = tokenizer(sentence, return_tensors="pt")
    with torch.no_grad():
        outputs = model(**inputs)
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
    bank_idx = tokens.index("bank")
    bank_vector = outputs.last_hidden_state[0, bank_idx, :]
    bank_vectors.append(bank_vector)
    print(f"Sentence: {sentence}")
    print(f"'bank' embedding (first 4 dims): {bank_vector[:4].numpy()}\n")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Sentence: She sat on the river bank.
'bank' embedding (first 4 dims): [-0.02623809 -0.5945933  -0.19887649 -0.09317101]

Sentence: He went to the bank to deposit his check.
'bank' embedding (first 4 dims): [ 0.72866297 -0.53897995 -0.10705479 -0.06346765]

Sentence: We had a picnic on the bank near the stream.
'bank' embedding (first 4 dims): [ 0.14970984 -0.27410126 -0.6775759   0.4769324 ]



In [10]:
vec1, vec2, vec3 = bank_vectors

cos_sim = F.cosine_similarity(vec1.unsqueeze(0), vec2.unsqueeze(0))

print("River bank vs. Financial bank:", cos_sim.item())

cos_sim = F.cosine_similarity(vec1.unsqueeze(0), vec3.unsqueeze(0))

print("Two uses of river bank", cos_sim.item())

River bank vs. Financial bank: 0.4898349940776825
Two uses of river bank 0.8119401931762695


In [11]:
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer("all-MiniLM-L6-v2")

s1 = "Winter mornings are cold and peaceful."
s2 = "Summer brings heat and sunshine."
s3 = "Inflation rose sharply last quarter."

embeddings = model.encode([s1, s2, s3])

print("s1 vs s2:", util.cos_sim(embeddings[0], embeddings[1]))
print("s1 vs s3:", util.cos_sim(embeddings[0], embeddings[2]))
print("s2 vs s3:", util.cos_sim(embeddings[1], embeddings[2]))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

s1 vs s2: tensor([[0.4993]])
s1 vs s3: tensor([[0.0019]])
s2 vs s3: tensor([[0.1032]])
